# 6. Evaluate the model, compare with baselines, predict new data

1. Run the model on every held-out test window and turn each softmax into a behaviour state:
   top-1 ≥ 0.70 → that state; a close pair (top-1 ≤ 0.70, top-2 ≥ 0.20, gap ≤ 0.40) → e.g. `bound_confined`;
   otherwise `unresolved_<top-1>`. Thresholds: `softmax_states:` in `configs/pipeline.yaml`.
2. Scores against the certain gate labels, confusion matrix, embedding.
3. Baselines on the same split (`python scripts/run_benchmarks.py`).
4. New data: `python scripts/predict.py --instant <instant_df_processed.csv> --windowed <time_windowed_df_processed.csv>`.

The paper numbers (590,814 held-out certain windows): accuracy 0.927, balanced accuracy 0.813, macro F1 0.750.

In [ ]:
import sys
sys.path.insert(0, "..")  # so `import sptnano` works from this folder

import polars as pl

from sptnano import config

paths = config.load_paths()
settings = config.load_settings()

from datetime import date

import pandas as pd
import torch

from sptnano import evaluate, predict
from sptnano.model import load_classifier

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Predictions on the held-out test windows

`RUN_PREDICTION = False` loads the saved paper predictions from `evaluation_dir`.
`True` predicts again (about 15 minutes on a GPU) and writes a new dated folder.

In [ ]:
RUN_PREDICTION = False

gated_windowed = paths["gated_dir"] / "windowed_df_hierarchical_gates.parquet"
gated_instant = paths["gated_dir"] / "instant_df_hierarchical_gates.parquet"

if RUN_PREDICTION:
    eval_dir = paths["training_output_dir"] / f"evaluation_block3_long_large_{date.today():%Y%m%d}"
    eval_dir.mkdir(parents=True, exist_ok=True)
    classifier, class_labels = load_classifier(paths["model_dir"], device)
    test_frames = predict.heldout_test_frames(paths["splits_dir"], gated_instant)
    predictions = predict.predict_windows(test_frames, classifier, class_labels, device=device)
    behavior_df = predict.add_softmax_states(predictions, class_labels, **settings["softmax_states"])
    behavior_df.to_parquet(eval_dir / "heldout_test_softmax_mixture_labels.parquet", index=False)

    predicted_uids = pl.DataFrame({"window_uid": behavior_df["window_uid"].astype(str).tolist()})
    for name, source in [("windows", gated_windowed), ("instant", gated_instant)]:
        table = pl.read_parquet(source).join(predicted_uids, on="window_uid", how="semi")
        predict.add_predictions(table, behavior_df).write_parquet(
            eval_dir / f"heldout_test_{name}_with_transformer_behavior_state.parquet"
        )
    print("Saved to", eval_dir)
else:
    eval_dir = paths["evaluation_dir"]
    behavior_df = pd.read_parquet(eval_dir / "heldout_test_softmax_mixture_labels.parquet")
    behavior_df["transformer_behavior_state"] = behavior_df["softmax_mixture_label"].astype(str)

behavior_df = evaluate.attach_gate_labels(behavior_df, gated_windowed)
print(behavior_df.shape)

## 2. Scores

In [ ]:
scores, report = evaluate.classification_summary(behavior_df)
print(evaluate.summary_line(scores))
display(report)

print("All gate labels (not only certain):")
evaluate.classification_summary(behavior_df, true_col="final_population")

display(evaluate.state_counts(behavior_df))
display(evaluate.mean_softmax_by_label(behavior_df))

In [ ]:
figure_dir = paths["plots_dir"] / "evaluation"
evaluate.plot_confusion_matrix_report(
    behavior_df[behavior_df["final_population_certain"].isin(evaluate.CLASS_ORDER)],
    true_col="final_population_certain",
    pred_col="pred_label",
    output_dir=figure_dir,
    name="heldout_certain_vs_pred",
    labels=evaluate.CLASS_ORDER,
    figsize=(5.4, 4.8),
)

## 3. Embedding

UMAP of the 16-D embedding on 50,000 windows (the paper figure version is in notebook 07).

In [ ]:
embedding = evaluate.compute_embedding_2d(behavior_df, method="umap", n_sample=50_000)
evaluate.plot_embedding(embedding, "umap_1", "umap_2", "pred_label", save_path=figure_dir / "umap_by_pred_label.png")

## 4. Baselines

Run once from a terminal (about an hour): `python scripts/run_benchmarks.py`.

- Random forest on the window scalars: an upper bound, because the gate labels are drawn on these scalars.
  The "no gate axes" version drops the four gate features but keeps correlated ones (step length, MSD).
- Random forest on the block3 tokens: the same raw-motion input the transformer gets.
- Gaussian HMM on per-frame motion, no labels used for fitting.

In [ ]:
benchmark_csv = paths["evaluation_dir"] / "benchmarks" / "benchmark_scores.csv"
rows = [{"method": "transformer_block3_long_large", **scores}]
if benchmark_csv.exists():
    rows += pd.read_csv(benchmark_csv).to_dict("records")
else:
    print("No benchmark scores yet: python scripts/run_benchmarks.py")
pd.DataFrame(rows)

## 5. New data

From a terminal:

    python scripts/predict.py --instant <folder>/saved_data/instant_df_processed.csv --windowed <folder>/saved_data/time_windowed_df_processed.csv

It writes `<folder>/saved_data/transformer_predictions/` with one row per window plus your frame and window
tables with the prediction columns added.